# Presidio Baseline: CRAPII + TAB

Entity schema, keep/redact policy, and metrics follow `EVALUATION_CONVENTIONS.md`.

In [ ]:
# Environment: pip install presidio-analyzer pandas matplotlib tqdm && python -m spacy download en_core_web_lg
import json, random, collections
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Configuration

In [ ]:
# Run size
QUICK_TEST = False                    # True: 10% of dev. False: all of dev and test.

# Data
DATA_DIR = next((p for p in [Path("../data/processed"), Path("./data/processed")] if p.exists()),
                Path("../data/processed"))
COMBINED_FILE = None                 # optional single JSON/JSONL file with a `split` field; overrides split files
SOURCES = ["crapii", "tab"]          # UNER excluded
SPLITS = ["train", "dev", "test"]
SAMPLE_FRAC = {"dev": 0.1} if QUICK_TEST else {"dev": 1.0, "test": 1.0}   # stratified by source
SEED = 13

# Outputs (gitignored)
OUTPUT_DIR = Path("./presidio_eval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Models
RUN_PRESIDIO = True
SPACY_MODEL = "en_core_web_lg"
LANGUAGE = "en"
PRESIDIO_SCORE_THRESHOLD = 0.35      # model confidence threshold (confidence_threshold in outputs); tuned on dev in 4.4
ADD_CUSTOM_RECOGNIZERS = False       # False = vanilla Presidio

# Entity schema and keep/redact policy (conventions section 1)
UNIFIED_LABELS = ["PERSON_PRIVATE", "PERSON_PUBLIC", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                  "DATE_OTHER", "LOCATION_OTHER", "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC", "ORG"]
KEEP_LABELS = ["PERSON_PUBLIC", "ORG", "DATE_OTHER", "LOCATION_OTHER"]
REDACT_LABELS = ["PERSON_PRIVATE", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                 "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC"]
assert sorted(KEEP_LABELS + REDACT_LABELS) == sorted(UNIFIED_LABELS)

# Scoring parameters (conventions section 2)
MATCH_THRESHOLD = 1.0                # span coverage required for a match (match_threshold in outputs)
BETA = 2.0  # F2 favors recall.

# Not specified in the conventions; pending group confirmation
EXCLUDE_IGNORED_SPANS = True         # TAB ignored_spans (MISC, QUANTITY) are not evaluated characters

## 2. Data

In [ ]:
def read_records(path):
    """JSON array or JSONL. JSONL is split on '\n' only (text may contain U+2028 and similar)."""
    raw = Path(path).read_text(encoding="utf-8-sig")
    if raw.lstrip().startswith("["):
        return json.loads(raw)
    records = []
    for line_number, line in enumerate(raw.split("\n"), start=1):
        if line.strip():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"{Path(path).name}, line {line_number}: {error}") from error
    return records

def load_corpus():
    if COMBINED_FILE is not None:
        records = read_records(COMBINED_FILE)
        if any("split" not in record for record in records):
            raise ValueError(f"{COMBINED_FILE}: every record needs a 'split' field")
    else:
        records = []
        for split in SPLITS:
            for record in read_records(DATA_DIR / f"{split}.jsonl"):
                record["split"] = split
                records.append(record)
    kept = [r for r in records if r.get("source") in SOURCES and r["split"] in SPLITS]
    print(f"loaded {len(records)} records, kept {len(kept)} ({', '.join(SOURCES)})")
    return kept

ALL_DOCS = load_corpus()
for doc in ALL_DOCS:
    doc["key"] = f'{doc["split"]}|{doc["doc_id"]}'
    doc["spans"] = doc.get("spans") or []
    doc["ignored_spans"] = doc.get("ignored_spans") or []
DOCS = {split: [d for d in ALL_DOCS if d["split"] == split] for split in SPLITS}

unknown_labels = {s["label"] for d in ALL_DOCS for s in d["spans"]} - set(UNIFIED_LABELS)
if unknown_labels:
    raise ValueError(f"labels outside the unified schema: {sorted(unknown_labels)}")

corpus_overview = pd.DataFrame([{
    "split": d["split"], "source": d["source"],
    "redact_spans": sum(s["label"] in REDACT_LABELS for s in d["spans"]),
    "keep_spans": sum(s["label"] in KEEP_LABELS for s in d["spans"]),
} for d in ALL_DOCS])
display(corpus_overview.groupby(["split", "source"]).agg(
    documents=("redact_spans", "size"),
    pii_documents=("redact_spans", lambda x: int((x > 0).sum())),
    redact_spans=("redact_spans", "sum"),
    keep_spans=("keep_spans", "sum")))

## 3. Detectors

In [ ]:
@dataclass
class Detector:
    name: str
    predict: Callable[[str], list]       # text -> [{"start", "end", "entity_type", "score"}], character offsets
    label_map: dict                      # model entity_type -> unified label
    default_threshold: float = 0.5
    fallback_label: Optional[Callable[[str], str]] = None   # for entity types missing from label_map

    def to_label(self, entity_type):
        if entity_type in self.label_map:
            return self.label_map[entity_type]
        return self.fallback_label(entity_type) if self.fallback_label else "OTHER_PII"

DETECTORS = {}
def register(detector):
    invalid = set(detector.label_map.values()) - set(UNIFIED_LABELS)
    if invalid:
        raise ValueError(f"{detector.name}: labels outside the unified schema: {invalid}")
    DETECTORS[detector.name] = detector
    print(f"registered '{detector.name}' (confidence threshold {detector.default_threshold})")

### 3.1 Presidio

In [ ]:
if RUN_PRESIDIO:
    from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
    from presidio_analyzer.nlp_engine import NlpEngineProvider

    nlp_config = {
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": LANGUAGE, "model_name": SPACY_MODEL}],
        "ner_model_configuration": {
            "model_to_presidio_entity_mapping": {
                "PER": "PERSON", "PERSON": "PERSON", "NORP": "NRP",
                "FAC": "LOCATION", "LOC": "LOCATION", "GPE": "LOCATION", "LOCATION": "LOCATION",
                "ORG": "ORGANIZATION", "ORGANIZATION": "ORGANIZATION",
                "DATE": "DATE_TIME", "TIME": "DATE_TIME",
            },
            "low_confidence_score_multiplier": 0.4,
            "low_score_entity_names": [],    # no ORG down-weighting
            "labels_to_ignore": ["CARDINAL", "EVENT", "LANGUAGE", "LAW", "MONEY",
                                 "ORDINAL", "PERCENT", "PRODUCT", "QUANTITY", "WORK_OF_ART"],
        },
    }
    analyzer = AnalyzerEngine(nlp_engine=NlpEngineProvider(nlp_configuration=nlp_config).create_engine(),
                              supported_languages=[LANGUAGE])

    if ADD_CUSTOM_RECOGNIZERS:
        analyzer.registry.add_recognizer(PatternRecognizer(          # ECHR application numbers
            supported_entity="ECHR_APP_NO", patterns=[Pattern("echr_app_no", r"\b\d{1,6}/\d{2}\b", 0.6)],
            context=["application", "applications", "no", "nos"]))
        analyzer.registry.add_recognizer(PatternRecognizer(          # social-media handles
            supported_entity="SOCIAL_HANDLE", patterns=[Pattern("handle", r"(?<!\w)@[A-Za-z0-9_]{3,30}\b", 0.5)]))

    PRESIDIO_TO_UNIFIED = {
        "PERSON": "PERSON_PRIVATE",      # Presidio does not separate private from public persons
        "LOCATION": "LOCATION_OTHER", "DATE_TIME": "DATE_OTHER", "ORGANIZATION": "ORG",
        "NRP": "DEMOGRAPHIC", "EMAIL_ADDRESS": "EMAIL", "PHONE_NUMBER": "PHONE", "URL": "URL_PERSONAL",
        "ECHR_APP_NO": "ID_NUM", "SOCIAL_HANDLE": "OTHER_PII",
    }
    ID_HINTS = ("SSN", "NUMBER", "LICENSE", "PASSPORT", "CARD", "IBAN", "CRYPTO", "NHS", "ITIN",
                "CODE", "_ID", "AADHAAR", "PAN", "NRIC", "ABN", "ACN", "TFN", "MEDICARE")

    def presidio_fallback_label(entity_type):
        return "ID_NUM" if any(hint in entity_type for hint in ID_HINTS) else "OTHER_PII"

    def presidio_predict(text):
        results = analyzer.analyze(text=text, language=LANGUAGE, score_threshold=0.0)
        return [{"start": r.start, "end": r.end, "entity_type": r.entity_type, "score": round(float(r.score), 4)}
                for r in results]

    register(Detector(
        name="presidio",
        predict=presidio_predict,
        label_map=PRESIDIO_TO_UNIFIED,
        default_threshold=PRESIDIO_SCORE_THRESHOLD,
        fallback_label=presidio_fallback_label,
    ))
    display(pd.Series({e: presidio_fallback_label(e) if e not in PRESIDIO_TO_UNIFIED else PRESIDIO_TO_UNIFIED[e]
                       for e in sorted(analyzer.get_supported_entities())}, name="unified_label"))

### 3.2 Additional detectors

In [ ]:
# Template. Add RUN_<MODEL> to section 1, map model labels to unified labels, return character offsets
# into the original text (chunk long documents and shift offsets), and return candidates down to a low score.
#
# if RUN_MODEL:
#     def model_predict(text):
#         return [{"start": ..., "end": ..., "entity_type": ..., "score": ...}]
#     register(Detector(name="model", predict=model_predict, label_map={...: "PERSON_PRIVATE"},
#                       default_threshold=0.5))

### 3.3 Run detectors

In [ ]:
def sample_docs():
    rng = random.Random(SEED)
    sampled = []
    for split in SPLITS:
        fraction = SAMPLE_FRAC.get(split, 0)
        if fraction <= 0:
            continue
        by_source = collections.defaultdict(list)
        for doc in DOCS[split]:
            by_source[doc["source"]].append(doc)
        for source, docs in sorted(by_source.items()):
            k = len(docs) if fraction >= 1 else max(1, round(len(docs) * fraction))
            sampled.extend(rng.sample(docs, k) if k < len(docs) else docs)
    return sampled

def run_detector(detector, docs):
    predictions, failures = {}, []
    for doc in tqdm(docs, desc=detector.name):
        try:
            predictions[doc["key"]] = detector.predict(doc["text"])
        except Exception as error:
            failures.append((doc["key"], repr(error)))
    print(f"[{detector.name}] {len(predictions)} docs predicted, {len(failures)} failures")
    return predictions, failures

assert DETECTORS, "no detectors registered"
SAMPLED_DOCS = sample_docs()
PREDS, FAILURES = {}, {}
for name, detector in DETECTORS.items():
    PREDS[name], FAILURES[name] = run_detector(detector, SAMPLED_DOCS)
MODELS = list(DETECTORS)
EVAL_DOCS = [d for d in SAMPLED_DOCS if all(d["key"] in PREDS[m] for m in MODELS)]   # scored by every model
display(pd.Series(collections.Counter((d["split"], d["source"]) for d in EVAL_DOCS), name="documents").sort_index())

## 4. Evaluation

In [ ]:
# Gold PII = spans with a REDACT label. Flagged = predictions above the confidence threshold whose
# unified label is in REDACT_LABELS. Character counts use non-whitespace characters; overlaps count once.

def char_mask(length, spans):
    mask = np.zeros(length, dtype=bool)
    for span in spans:
        mask[max(0, span["start"]):min(length, span["end"])] = True
    return mask

def flatten_longest_first(spans):
    """Count nested or overlapping gold PII spans once: keep the longest, drop spans overlapping it."""
    kept = []
    for span in sorted(spans, key=lambda s: (-(s["end"] - s["start"]), s["start"])):
        if all(span["end"] <= k["start"] or span["start"] >= k["end"] for k in kept):
            kept.append(span)
    return kept

_GOLD_VIEWS = {}
def gold_view(doc):
    """Prediction-independent masks per document, reused across models and thresholds."""
    if doc["key"] not in _GOLD_VIEWS:
        text = doc["text"]
        length = len(text)
        non_whitespace = np.fromiter((not ch.isspace() for ch in text), dtype=bool, count=length)
        redact_spans = [s for s in doc["spans"] if s["label"] in REDACT_LABELS]
        pii = char_mask(length, redact_spans) & non_whitespace
        evaluated = non_whitespace.copy()
        if EXCLUDE_IGNORED_SPANS:
            evaluated &= ~(char_mask(length, doc["ignored_spans"]) & ~pii)
        keep = {label: char_mask(length, [s for s in doc["spans"] if s["label"] == label]) & evaluated & ~pii
                for label in KEEP_LABELS}
        pii_spans = [s for s in flatten_longest_first(redact_spans) if evaluated[s["start"]:s["end"]].any()]
        _GOLD_VIEWS[doc["key"]] = {"evaluated": evaluated, "pii": pii, "keep": keep, "pii_spans": pii_spans}
    return _GOLD_VIEWS[doc["key"]]

def flagged_predictions(predictions, detector, threshold):
    flagged, seen_offsets = [], set()
    for p in sorted(predictions, key=lambda p: -p["score"]):
        label = detector.to_label(p["entity_type"])
        if p["score"] < threshold or label not in REDACT_LABELS or (p["start"], p["end"]) in seen_offsets:
            continue
        seen_offsets.add((p["start"], p["end"]))
        flagged.append({**p, "label": label})
    return flagged

def evaluate_document(doc, flagged):
    gold = gold_view(doc)
    evaluated, pii = gold["evaluated"], gold["pii"]
    flagged_mask = char_mask(len(doc["text"]), flagged) & evaluated
    uncovered_pii = pii & ~flagged_mask
    flagged_non_pii = flagged_mask & ~pii

    document_row = {
        "flagged_character_count": int(flagged_mask.sum()),
        "flagged_pii_character_count": int((flagged_mask & pii).sum()),
        "annotated_pii_character_count": int(pii.sum()),
        "uncovered_pii_character_count": int(uncovered_pii.sum()),
        "pii_document": bool(pii.any()),
        "leaking_pii_document": bool(uncovered_pii.any()),
    }
    remaining = flagged_non_pii.copy()      # attribute over-redacted characters to keep labels, then unlabeled
    for label in KEEP_LABELS:
        on_label = remaining & gold["keep"][label]
        document_row[f"over_redacted_{label}"] = int(on_label.sum())
        remaining &= ~on_label
    document_row["over_redacted_UNLABELED"] = int(remaining.sum())

    pii_span_rows = []
    for span in gold["pii_spans"]:
        span_evaluated = evaluated[span["start"]:span["end"]]
        coverage = (flagged_mask[span["start"]:span["end"]] & span_evaluated).sum() / span_evaluated.sum()
        pii_span_rows.append({"label": span["label"], "text": doc["text"][span["start"]:span["end"]],
                              "coverage": coverage, "fully_covered": coverage >= MATCH_THRESHOLD})

    predicted_span_rows = []
    for p in flagged:
        span_evaluated = evaluated[p["start"]:p["end"]]
        if not span_evaluated.any():
            continue
        pii_share = (pii[p["start"]:p["end"]] & span_evaluated).sum() / span_evaluated.sum()
        predicted_span_rows.append({"entity_type": p["entity_type"], "label": p["label"], "score": p["score"],
                                    "text": doc["text"][p["start"]:p["end"]], "pii_only": pii_share >= MATCH_THRESHOLD})
    return document_row, pii_span_rows, predicted_span_rows

def evaluate(docs, model, threshold=None):
    detector = DETECTORS[model]
    threshold = detector.default_threshold if threshold is None else threshold
    rows = {"documents": [], "pii_spans": [], "predicted_spans": []}
    for doc in docs:
        tag = {"model": model, "confidence_threshold": threshold, "match_threshold": MATCH_THRESHOLD, "split": doc["split"], "source": doc["source"], "key": doc["key"]}
        flagged = flagged_predictions(PREDS[model][doc["key"]], detector, threshold)
        document_row, pii_span_rows, predicted_span_rows = evaluate_document(doc, flagged)
        rows["documents"].append({**tag, **document_row})
        rows["pii_spans"].extend({**tag, **r} for r in pii_span_rows)
        rows["predicted_spans"].extend({**tag, **r} for r in predicted_span_rows)
    columns = {"pii_spans": ["label", "text", "coverage", "fully_covered"],
               "predicted_spans": ["entity_type", "label", "score", "text", "pii_only"]}
    return {name: pd.DataFrame(r) if r else pd.DataFrame(columns=list(tag) + columns.get(name, []))
            for name, r in rows.items()}

def evaluate_all(docs, threshold=None):
    parts = [evaluate(docs, model, threshold) for model in MODELS]
    return {name: pd.concat([p[name] for p in parts], ignore_index=True) for name in parts[0]}

def span_f_beta(span_precision, span_recall, beta=BETA):
    if span_recall == 0:
        return 0.0
    if np.isnan(span_precision) or np.isnan(span_recall):
        return np.nan
    return (1 + beta**2) * span_precision * span_recall / (beta**2 * span_precision + span_recall)

HEADLINE_METRICS = ["span_precision", "span_recall", "span_f2"]
DEVELOPMENT_METRICS = ["character_over_redaction_percentage", "character_leakage_percentage", "document_leakage_rate"]

def summarize(results, by=("model", "split", "source")):
    by = list(by)
    documents = results["documents"].groupby(by).agg(
        confidence_threshold=("confidence_threshold", "first"), match_threshold=("match_threshold", "first"),
        documents=("key", "size"),
        pii_document_count=("pii_document", "sum"), leaking_pii_document_count=("leaking_pii_document", "sum"),
        flagged_character_count=("flagged_character_count", "sum"),
        flagged_pii_character_count=("flagged_pii_character_count", "sum"),
        annotated_pii_character_count=("annotated_pii_character_count", "sum"),
        uncovered_pii_character_count=("uncovered_pii_character_count", "sum"))
    pii_spans = results["pii_spans"].groupby(by).agg(
        evaluated_pii_span_count=("fully_covered", "size"), fully_covered_pii_span_count=("fully_covered", "sum"))
    predicted_spans = results["predicted_spans"].groupby(by).agg(
        evaluated_predicted_span_count=("pii_only", "size"), pii_only_predicted_span_count=("pii_only", "sum"))
    table = documents.join(pii_spans, how="left").join(predicted_spans, how="left").fillna(0)
    count_columns = ["documents", "pii_document_count", "evaluated_pii_span_count", "evaluated_predicted_span_count"]
    table[count_columns] = table[count_columns].astype(int)

    def ratio(numerator, denominator):
        return table[numerator] / table[denominator].replace(0, np.nan)

    table["span_precision"] = ratio("pii_only_predicted_span_count", "evaluated_predicted_span_count")
    table["span_recall"] = ratio("fully_covered_pii_span_count", "evaluated_pii_span_count")
    table["span_f2"] = [span_f_beta(p, r) for p, r in zip(table.span_precision, table.span_recall)]
    table["character_over_redaction_percentage"] = 100 * (
        table.flagged_character_count - table.flagged_pii_character_count) / table.flagged_character_count.replace(0, np.nan)
    table["character_leakage_percentage"] = 100 * ratio("uncovered_pii_character_count", "annotated_pii_character_count")
    table["document_leakage_rate"] = 100 * ratio("leaking_pii_document_count", "pii_document_count")
    return table.reset_index()[by + ["confidence_threshold", "match_threshold"] + count_columns + HEADLINE_METRICS + DEVELOPMENT_METRICS]

def show_metrics(table):
    formats = {**{m: "{:.3f}" for m in HEADLINE_METRICS}, **{m: "{:.1f}" for m in DEVELOPMENT_METRICS},
               "confidence_threshold": "{:.2f}", "match_threshold": "{:.2f}"}
    display(table.style.format({k: v for k, v in formats.items() if k in table.columns}, na_rep="-"))

### 4.1 Results at default threshold

In [ ]:
RESULTS = evaluate_all(EVAL_DOCS)
summary = summarize(RESULTS)
show_metrics(summary)

### 4.2 Span recall by gold label

In [ ]:
recall_by_label = (RESULTS["pii_spans"].groupby(["model", "split", "source", "label"])
                   .agg(evaluated_pii_span_count=("fully_covered", "size"), span_recall=("fully_covered", "mean"),
                        mean_coverage=("coverage", "mean"))
                   .reset_index())
display(recall_by_label.style.format({"span_recall": "{:.3f}", "mean_coverage": "{:.3f}"}))

### 4.3 Over-redaction breakdown

In [ ]:
# Share of flagged characters (percent) that fell on each keep label or on unlabeled text
over_redaction_columns = [f"over_redacted_{label}" for label in KEEP_LABELS] + ["over_redacted_UNLABELED"]
over_redaction = RESULTS["documents"].groupby(["model", "split", "source"])[
    ["flagged_character_count"] + over_redaction_columns].sum()
over_redaction_breakdown = (100 * over_redaction[over_redaction_columns]
                            .div(over_redaction.flagged_character_count.replace(0, np.nan), axis=0))
over_redaction_breakdown.columns = [c.replace("over_redacted_", "") for c in over_redaction_columns]
display(over_redaction_breakdown.round(1))

# Span precision by model entity type
precision_by_entity = (RESULTS["predicted_spans"].groupby(["model", "split", "source", "entity_type"])
                       .agg(evaluated_predicted_span_count=("pii_only", "size"), span_precision=("pii_only", "mean"),
                            median_score=("score", "median"))
                       .reset_index())
display(precision_by_entity.style.format({"span_precision": "{:.3f}", "median_score": "{:.2f}"}))

### 4.4 Confidence threshold: tuned on dev, applied to test

In [ ]:
# Selection: highest dev span F2 per model and source; ties go to the higher confidence threshold.
THRESHOLDS = np.round(np.arange(0.0, 1.0, 0.05), 2)
dev_docs = [d for d in EVAL_DOCS if d["split"] == "dev"]
sweep = pd.concat([summarize(evaluate(dev_docs, model, threshold), by=("model", "source"))
                   for model in MODELS for threshold in tqdm(THRESHOLDS, desc=f"sweep {model}")],
                  ignore_index=True)

tuned_thresholds = (sweep.dropna(subset=["span_f2"])
                    .sort_values(["span_f2", "confidence_threshold"], ascending=[False, False])
                    .groupby(["model", "source"]).head(1)
                    .sort_values(["model", "source"]).reset_index(drop=True))
show_metrics(tuned_thresholds)

panels = list(sweep.groupby(["model", "source"]))
fig, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 4), squeeze=False)
for ax, ((model, source), table) in zip(axes.flat, panels):
    for metric in HEADLINE_METRICS:
        ax.plot(table.confidence_threshold, table[metric], marker=".", label=metric)
    for metric in DEVELOPMENT_METRICS:
        ax.plot(table.confidence_threshold, table[metric] / 100, ls="--", label=f"{metric} / 100")
    chosen = tuned_thresholds[(tuned_thresholds.model == model) & (tuned_thresholds.source == source)]
    if len(chosen):
        ax.axvline(chosen.confidence_threshold.iloc[0], c="grey", ls=":")
    ax.set(title=f"{model} | {source} | dev", xlabel="confidence threshold", ylim=(0, 1))
    ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

test_docs = [d for d in EVAL_DOCS if d["split"] == "test"]
tuned_test = pd.DataFrame()
tuned_parts = [summarize(evaluate([d for d in test_docs if d["source"] == row.source], row.model, row.confidence_threshold))
               for row in tuned_thresholds.itertuples() if any(d["source"] == row.source for d in test_docs)]
if tuned_parts:
    tuned_test = pd.concat(tuned_parts, ignore_index=True)
    show_metrics(tuned_test)

## 5. Exports

In [ ]:
metrics_summary = pd.concat([summary.assign(setting="default threshold"),
                             tuned_test.assign(setting="dev-tuned threshold")], ignore_index=True)
exports = {
    "metrics_summary.csv": metrics_summary,
    "recall_by_label.csv": recall_by_label,
    "over_redaction_breakdown.csv": over_redaction_breakdown.reset_index(),
    "precision_by_entity.csv": precision_by_entity,
    "threshold_sweep_dev.csv": sweep,
    "tuned_thresholds_dev.csv": tuned_thresholds,
    "pii_span_results.csv": RESULTS["pii_spans"],
    "predicted_span_results.csv": RESULTS["predicted_spans"],
}
for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
print("wrote:", ", ".join(exports))